## HEAT MAP
Ce code réalise le **tracé de la trajectoire du regard** de l’utilisateur sur une affiche,  
puis générer une **carte de chaleur** afin de visualiser les zones les plus observées.

In [ ]:
import plotly.graph_objects as go
import numpy as np
from PIL import Image
from heat_map import *
from heat_map_utils import *

In [ ]:
# Paramètres et données
nom_affiche = 'Gautier'

# Charger l'image
affiche_path = f'./data/Affiches/{nom_affiche}.png'
poster = Image.open(affiche_path)
W, H = poster.size

# Charger des points de test
x, y = get_test_set(W, H)
x, y = traitement_points(x, y, W, H)

In [ ]:
# Calculer la densité
z = heat_map_density(x, y, W, H)

show_points_on_poster(affiche_path, x, y)

show_heat_map_on_poster(affiche_path, z)


## Appelle avec une seule fonction

In [ ]:
x, y = get_test_set(W, H)
nom = "Bell.png"

step_heat_map(x, y, nom)

## Somme des heat maps

In [ ]:
# Sommer les points de toutes les vidéo pour tous les posters dans un seul dictionnaire
positions_all_posters_total = np.load(f"output/posters_0.npy", allow_pickle=True).item()
for sujet_id in range(1, 5):
    positions_all_posters = np.load(f"output/posters_{sujet_id}.npy", allow_pickle=True).item()
    for key in positions_all_posters.keys():
        positions_all_posters_total[key][0].extend(positions_all_posters[key][0])
        positions_all_posters_total[key][1].extend(positions_all_posters[key][0])

for nom in positions_all_posters_total.keys():
    x = positions_all_posters_total[nom][0]
    y = positions_all_posters_total[nom][1]
    step_heat_map(x, y, nom, trace=False)



In [ ]:
import plotly.graph_objects as go
import numpy as np

colorscale = [
    [0.0, "rgba(255,255,255,0.0)"],
    [0.2, "rgba(0,0,255,0.4)"],
    [0.4, "rgba(0,255,0,0.8)"],
    [0.6, "rgba(255,255,0,0.8)"],
    [0.8, "rgba(255,165,0,0.8)"],
    [1.0, "rgba(255,0,0,0.8)"]
]

# Données fictives juste pour afficher la colorbar
z = np.linspace(0, 1, 100).reshape(100, 1)

fig = go.Figure(
    data=go.Heatmap(
        z=z,
        colorscale=colorscale,
        showscale=True,
        colorbar=dict(
            title="Intensité",
            tickvals=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
            ticktext=["0", "0.2", "0.4", "0.6", "0.8", "1.0"],
            len=0.9
        )
    )
)

fig.update_layout(
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    width=300,
    height=500
)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ======================
# Colorscale fournie
# ======================
colorscale = [
    [0.0, "rgba(255,255,255,0.0)"],
    [0.2, "rgba(0,0,255,0.4)"],
    [0.4, "rgba(0,255,0,0.8)"],
    [0.6, "rgba(255,255,0,0.8)"],
    [0.8, "rgba(255,165,0,0.8)"],
    [1.0, "rgba(255,0,0,0.8)"]
]

# ======================
# Grille commune
# ======================
N = 100
x, y = np.meshgrid(np.linspace(-1, 1, N), np.linspace(-1, 1, N))
r = np.sqrt(x**2 + y**2)

# ======================
# 1. Carré uniforme 50x50
# ======================
square_uniform = np.zeros((N, N))
start = N // 2 - 25
end = N // 2 + 25
square_uniform[start:end, start:end] = 1.0

# ======================
# 2. Cercle uniforme
# ======================
circle_uniform = np.zeros((N, N))
circle_uniform[r <= 0.5] = 1.0

# ======================
# 3. Carré basé sur la distance au centre
# ======================
square_distance = np.zeros((N, N))

mask_square = (
    (np.abs(x) <= 0.5) &
    (np.abs(y) <= 0.5)
)

d = np.maximum(np.abs(x), np.abs(y))  # distance de Chebyshev
square_distance[mask_square] = 1 - d[mask_square] / 0.5
square_distance /= square_distance.max()

# ======================
# 4. Filtre gaussien
# ======================
sigma = 0.25
gaussian = np.exp(-(r**2) / (2 * sigma**2))
gaussian /= gaussian.max()

# ======================
# Affichage Plotly
# ======================
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[
        "Carré uniforme",
        "Cercle uniforme",
        "Carré – distance au centre",
        "Filtre gaussien"
    ]
)

fig.add_trace(go.Heatmap(
    z=square_uniform,
    colorscale=colorscale,
    showscale=False
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=circle_uniform,
    colorscale=colorscale,
    showscale=False
), row=1, col=2)

fig.add_trace(go.Heatmap(
    z=square_distance,
    colorscale=colorscale,
    showscale=False
), row=1, col=3)

fig.add_trace(go.Heatmap(
    z=gaussian,
    colorscale=colorscale,
    showscale=True,
    colorbar=dict(title="Intensité")
), row=1, col=4)

# ======================
# Affichage orthonormé
# ======================
fig.update_xaxes(
    visible=False,
    scaleanchor="y",
    scaleratio=1
)
fig.update_yaxes(
    visible=False,
    scaleanchor="x",
    scaleratio=1
)

fig.update_layout(
    width=1200,
    height=350,
    title="Forme de la réponse dans la heatmap"
)

fig.show()
